# **DLT Pipeline**

### **Streaming Table**

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Expectations
my_rules = {
    "rule1": "product_id IS NOT NULL",
    "rule2": "product_name IS NOT NULL"
}

In [0]:
# Batch/ Full refresh read on a table - creating a mv

@dlt.table(name = "gold_products_stage")
@dlt.expect_all_or_drop(my_rules)
def gold_products_stage():
    df_products =  spark.readStream.table("databricks_ete_project.silver.silver_products")
    df_products = df_products.drop('_rescued_data')
    return df_products

### **Apply Changes API**

In [0]:
# main gold_products table by

dlt.create_streaming_table(name = "gold_products")

In [0]:
dlt.apply_changes(
  target = "gold_products",
  source = "gold_products_stage",
  keys = ["product_id"],
  sequence_by = "product_id", # here actually a date column is required
  stored_as_scd_type = 2
)

In [0]:
# from pyspark import pipelines as dp
# from pyspark.sql.functions import *
# from pyspark.sql.types import *

In [0]:
# Expectations
# my_rules = {
#     "rule1": "product_id IS NOT NULL",
#     "rule2": "product_name IS NOT NULL"
# }

In [0]:
# Streaming read with skipChangeCommits - creating a temporary view

# @dp.temporary_view()
# @dp.expect_all_or_drop(my_rules)
# def gold_products_stage():
#     df_products = spark.readStream.option("skipChangeCommits", "true").table("databricks_ete_project.silver.silver_products")
#     # df_products = df_products.drop('_rescued_data')
#     return df_products

In [0]:
# main gold_products table

# dp.create_streaming_table(name = "gold_products")

In [0]:
# dp.create_auto_cdc_flow(
#   target = "gold_products",
#   source = "gold_products_stage",
#   keys = ["product_id"],
#   sequence_by = "product_id", # here actually a date column is required
#   stored_as_scd_type = 2
# )